<h1 style="font-family: Georgia, serif; color: #1a1a2e; border-bottom: 3px solid #c0392b; padding-bottom: 12px; margin-bottom: 20px;">
  Global Sovereign Debt Crisis Monitor (2000-Present)
</h1>
<p style="font-size: 15px; color: #444; max-width: 860px; line-height: 1.7;">
  This notebook pulls data from the IMF DataMapper API, World Bank Open Data API, and FRED
  to build a yearly country-level panel covering sovereign debt levels, fiscal health, and
  early-warning risk signals. It runs on a schedule via GitHub Actions and keeps the Kaggle
  dataset up to date automatically.
</p>
<table style="border-collapse: collapse; font-size: 13px; margin-top: 18px; min-width: 620px;">
  <tr style="background: #1a1a2e; color: white;">
    <th style="padding: 9px 14px; text-align: left;">Source</th>
    <th style="padding: 9px 14px; text-align: left;">What it covers</th>
    <th style="padding: 9px 14px; text-align: left;">Auth needed</th>
    <th style="padding: 9px 14px; text-align: left;">Update cadence</th>
  </tr>
  <tr style="background: #f8f9fa;">
    <td style="padding: 8px 14px; border: 1px solid #ddd;">IMF WEO (DataMapper API)</td>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">Debt/GDP, fiscal balance, revenue, interest payments</td>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">None</td>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">Bi-annual (Apr/Oct)</td>
  </tr>
  <tr>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">World Bank Open Data</td>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">External debt, reserves, GDP, population, region/income group</td>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">None</td>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">Annual</td>
  </tr>
  <tr style="background: #f8f9fa;">
    <td style="padding: 8px 14px; border: 1px solid #ddd;">FRED (St. Louis Fed)</td>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">US Treasury yields, US debt-to-GDP</td>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">Free API key</td>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">Daily/Quarterly</td>
  </tr>
  <tr>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">Manual reference data</td>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">S&amp;P ratings, sovereign defaults, IMF program flags</td>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">N/A</td>
    <td style="padding: 8px 14px; border: 1px solid #ddd;">Maintained in notebook</td>
  </tr>
</table>

<h2 style="font-family: Georgia, serif; color: #1a1a2e; border-left: 4px solid #c0392b; padding-left: 12px;">Setup</h2>

In [ ]:
import os
import time
import json
import textwrap
from datetime import datetime

import requests
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

# year range - goes up to the current year so new IMF/WB vintages are picked up automatically
START_YEAR = 2000
END_YEAR   = datetime.now().year

# IMF DataMapper series codes -> column names in the final dataset
IMF_SERIES = {
    'govt_debt_pct_gdp':        'GGXWDG_NGDP',  # gross govt debt as % of GDP
    'govt_debt_usd_bn':         'GGXWDG',        # gross govt debt in USD billions
    'fiscal_balance_pct_gdp':   'GGXCNL_NGDP',  # net lending/borrowing as % of GDP
    'primary_balance_pct_gdp':  'GGXONLB_NGDP', # primary balance (excl. interest) as % GDP
    'govt_revenue_pct_gdp':     'GGR_NGDP',      # govt revenue as % of GDP
    'interest_payments_pct_gdp':'GGXI_NGDP',     # interest payments as % of GDP
    '_gdp_usd_bn':              'NGDPD',          # nominal GDP USD bn (for derived calcs)
    '_population_mn':           'LP',             # population in millions (for per-capita)
}

# World Bank indicator codes -> column names
WB_SERIES = {
    'external_debt_usd_bn':  'DT.DOD.DECT.CD',  # total external debt stocks, current USD
    'foreign_reserves_usd_bn':'FI.RES.TOTL.CD', # total foreign reserves, current USD
    '_gdp_wb_usd':           'NY.GDP.MKTP.CD',   # GDP current USD (WB version)
    '_population_wb':        'SP.POP.TOTL',       # total population
}

# FRED API key - set as a GitHub secret called FRED_API_KEY
# if it's missing, FRED data is skipped gracefully
FRED_KEY = os.environ.get('FRED_API_KEY', '')

print(f'Year range: {START_YEAR} - {END_YEAR}')
print(f'FRED key present: {bool(FRED_KEY)}')
print('All imports OK')

<h2 style="font-family: Georgia, serif; color: #1a1a2e; border-left: 4px solid #c0392b; padding-left: 12px;">1. IMF World Economic Outlook Data</h2>
<p style="color: #555; font-size: 14px; max-width: 820px;">
  Uses the IMF DataMapper API - no key required, completely free. Each GET request returns
  a full time-series for all countries for one indicator. We pull the eight series we need
  and merge them on (iso3_code, year).
</p>
<ul style="color: #555; font-size: 14px;">
  <li>Endpoint: <code>https://www.imf.org/external/datamapper/api/v1/{SERIES_CODE}</code></li>
  <li>No pagination needed - the response has every country and year in one shot</li>
  <li>We add a short sleep between requests to be polite to the server</li>
</ul>

In [ ]:
def fetch_imf_series(series_code: str, col_name: str) -> pd.DataFrame:
    """Pull one IMF DataMapper series for all countries and years."""
    url = f'https://www.imf.org/external/datamapper/api/v1/{series_code}'
    try:
        resp = requests.get(url, timeout=90)
        resp.raise_for_status()
        raw = resp.json().get('values', {}).get(series_code, {})
    except Exception as exc:
        print(f'    WARNING: could not fetch {series_code}: {exc}')
        # return an empty frame with the right columns so merges still work
        return pd.DataFrame(columns=['iso3_code', 'year', col_name])

    rows = []
    for iso3, year_vals in raw.items():
        # skip aggregate/regional codes - real country codes are exactly 3 alpha chars
        if len(iso3) != 3 or not iso3.isalpha():
            continue
        for yr_str, val in year_vals.items():
            try:
                yr = int(yr_str)
            except ValueError:
                continue
            if START_YEAR <= yr <= END_YEAR and val is not None:
                rows.append({'iso3_code': iso3.upper(), 'year': yr, col_name: float(val)})

    if not rows:
        print(f'    WARNING: {series_code} returned no rows')
        return pd.DataFrame(columns=['iso3_code', 'year', col_name])
    return pd.DataFrame(rows)


def safe_merge_frames(frames: dict) -> pd.DataFrame:
    """
    Merge a dict of DataFrames on (iso3_code, year).
    Skips any frame that is empty or missing the key columns.
    """
    result = None
    for col_name, df in frames.items():
        # skip empty frames - they would wipe out keys in an outer merge
        if df.empty or 'iso3_code' not in df.columns or 'year' not in df.columns:
            print(f'    skipping empty frame: {col_name}')
            continue
        if result is None:
            result = df
        else:
            result = result.merge(df, on=['iso3_code', 'year'], how='outer')
    if result is None:
        raise RuntimeError('All IMF series came back empty - check network access')
    return result


print('Fetching IMF DataMapper series...')
imf_frames = {}
for col_name, series_code in IMF_SERIES.items():
    print(f'  {series_code:20s} -> {col_name}')
    imf_frames[col_name] = fetch_imf_series(series_code, col_name)
    time.sleep(0.6)  # light throttle - IMF server is not huge

imf_panel = safe_merge_frames(imf_frames)
imf_panel = imf_panel.sort_values(['iso3_code', 'year']).reset_index(drop=True)

print(f'\nIMF panel: {len(imf_panel):,} rows | {imf_panel["iso3_code"].nunique()} countries')
imf_panel.head()

<h2 style="font-family: Georgia, serif; color: #1a1a2e; border-left: 4px solid #c0392b; padding-left: 12px;">2. World Bank Open Data</h2>
<p style="color: #555; font-size: 14px; max-width: 820px;">
  The World Bank API gives us external debt, foreign reserves, country metadata (region,
  income group), and a cross-check for GDP and population. Also no key required.
</p>
<ul style="color: #555; font-size: 14px;">
  <li>Endpoint: <code>https://api.worldbank.org/v2/country/all/indicator/{CODE}?format=json</code></li>
  <li>The API paginates results - we loop through all pages</li>
  <li>Country metadata (region / income group) comes from a separate endpoint</li>
</ul>

In [ ]:
def fetch_wb_country_metadata() -> pd.DataFrame:
    """Get country name, ISO3, World Bank region, and income group."""
    url = 'https://api.worldbank.org/v2/country?format=json&per_page=300'
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()

    rows = []
    for c in resp.json()[1] or []:
        # skip aggregate groups (they have region id 'NA')
        if c.get('region', {}).get('id') == 'NA':
            continue
        rows.append({
            'iso3_code':    c.get('id', '').upper(),
            'country':      c.get('name', ''),
            'region':       c.get('region', {}).get('value', ''),
            'income_group': c.get('incomeLevel', {}).get('value', ''),
        })
    return pd.DataFrame(rows)


def fetch_wb_indicator(indicator: str, col_name: str) -> pd.DataFrame:
    """Fetch one World Bank indicator for all countries and years, handling pagination."""
    rows = []
    page = 1
    try:
        while True:
            url = (
                f'https://api.worldbank.org/v2/country/all/indicator/{indicator}'
                f'?format=json&per_page=1000&date={START_YEAR}:{END_YEAR}&page={page}'
            )
            resp = requests.get(url, timeout=90)
            resp.raise_for_status()
            payload = resp.json()
            if len(payload) < 2 or not payload[1]:
                break
            for item in payload[1]:
                if item.get('value') is None:
                    continue
                iso3 = (item.get('countryiso3code') or '').upper()
                if len(iso3) != 3:
                    continue
                rows.append({'iso3_code': iso3, 'year': int(item['date']), col_name: item['value']})
            total_pages = payload[0].get('pages', 1)
            if page >= total_pages:
                break
            page += 1
            time.sleep(0.3)
    except Exception as exc:
        print(f'    WARNING: could not fetch WB {indicator}: {exc}')

    if not rows:
        return pd.DataFrame(columns=['iso3_code', 'year', col_name])
    return pd.DataFrame(rows)


print('Fetching World Bank country metadata...')
wb_meta = fetch_wb_country_metadata()
print(f'  {len(wb_meta)} countries loaded')

print('Fetching World Bank indicator series...')
wb_frames = {}
for col_name, indicator in WB_SERIES.items():
    print(f'  {indicator:30s} -> {col_name}')
    wb_frames[col_name] = fetch_wb_indicator(indicator, col_name)
    time.sleep(0.6)

wb_panel = safe_merge_frames(wb_frames)

# convert raw USD values to billions
if 'external_debt_usd_bn' in wb_panel.columns:
    wb_panel['external_debt_usd_bn']    = wb_panel['external_debt_usd_bn']    / 1e9
if 'foreign_reserves_usd_bn' in wb_panel.columns:
    wb_panel['foreign_reserves_usd_bn'] = wb_panel['foreign_reserves_usd_bn'] / 1e9
if '_gdp_wb_usd' in wb_panel.columns:
    wb_panel['_gdp_wb_usd_bn'] = wb_panel['_gdp_wb_usd'] / 1e9
    wb_panel = wb_panel.drop(columns=['_gdp_wb_usd'])

wb_panel = wb_panel.sort_values(['iso3_code', 'year']).reset_index(drop=True)
print(f'\nWB panel: {len(wb_panel):,} rows | {wb_panel["iso3_code"].nunique()} countries')
wb_panel.head()

<h2 style="font-family: Georgia, serif; color: #1a1a2e; border-left: 4px solid #c0392b; padding-left: 12px;">3. FRED Data (US-specific)</h2>
<p style="color: #555; font-size: 14px; max-width: 820px;">
  FRED gives us US-specific context: 10-year Treasury yield and the federal debt-to-GDP ratio.
  These are joined into the panel only for the USA row.
  This step is skipped if <code>FRED_API_KEY</code> is not set.
</p>

In [ ]:
def fetch_fred_series(series_id: str, col_name: str, api_key: str,
                      agg: str = 'avg') -> pd.DataFrame:
    """
    Pull an annual average (or end-of-year) value from FRED.
    agg: 'avg' or 'eop' (end of period)
    """
    url = (
        f'https://api.stlouisfed.org/fred/series/observations'
        f'?series_id={series_id}'
        f'&api_key={api_key}'
        f'&file_type=json'
        f'&observation_start={START_YEAR}-01-01'
        f'&frequency=a'
        f'&aggregation_method={agg}'
    )
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    rows = []
    for obs in resp.json().get('observations', []):
        try:
            val = float(obs['value'])
            rows.append({'year': int(obs['date'][:4]), col_name: val})
        except (ValueError, KeyError):
            pass
    df = pd.DataFrame(rows)
    df['iso3_code'] = 'USA'
    return df


fred_panel = pd.DataFrame()

if FRED_KEY:
    print('FRED key found - fetching US series...')
    fred_dfs = []
    # 10-year Treasury yield (annual average)
    fred_dfs.append(fetch_fred_series('DGS10', 'us_10yr_yield_pct', FRED_KEY, agg='avg'))
    time.sleep(0.5)
    # Federal debt as % of GDP (quarterly -> annual)
    fred_dfs.append(fetch_fred_series('GFDEGDQ188S', 'us_federal_debt_pct_gdp_fred', FRED_KEY, agg='avg'))

    fred_panel = fred_dfs[0]
    for df in fred_dfs[1:]:
        fred_panel = fred_panel.merge(df, on=['iso3_code', 'year'], how='outer')

    print(f'FRED panel: {len(fred_panel)} rows')
else:
    print('No FRED_API_KEY found - skipping US-specific FRED data.')
    print('Set FRED_API_KEY as a GitHub secret to enable this.  Get a free key at:')
    print('  https://fred.stlouisfed.org/docs/api/api_key.html')

<h2 style="font-family: Georgia, serif; color: #1a1a2e; border-left: 4px solid #c0392b; padding-left: 12px;">4. Static Reference Data</h2>
<p style="color: #555; font-size: 14px; max-width: 820px;">
  Three sets of data that cannot be scraped from a clean API and must be maintained manually:
</p>
<ul style="color: #555; font-size: 14px;">
  <li><b>S&amp;P sovereign credit ratings</b> - representative current ratings by country, mapped to a numeric scale (AAA=21 ... D=0) for ML use</li>
  <li><b>Sovereign default events</b> - known default years compiled from public records and IMF reports</li>
  <li><b>IMF program flags</b> - known years a country was under an active IMF loan program</li>
</ul>
<p style="color: #888; font-size: 13px; font-style: italic;">
  Update these tables when new events happen - they do not refresh automatically.
</p>

In [ ]:
# ---- S&P numeric rating scale ----
RATING_MAP = {
    'AAA': 21, 'AA+': 20, 'AA': 19, 'AA-': 18,
    'A+': 17,  'A': 16,   'A-': 15,
    'BBB+': 14,'BBB': 13, 'BBB-': 12,
    'BB+': 11, 'BB': 10,  'BB-': 9,
    'B+': 8,   'B': 7,    'B-': 6,
    'CCC+': 5, 'CCC': 4,  'CCC-': 3,
    'CC': 2,   'C': 1,    'D': 0, 'SD': 0,
    'NR': np.nan,
}

# representative current S&P rating per country
# source: S&P Global Ratings / Wikipedia sovereign ratings tables
CURRENT_SP_RATINGS = {
    'USA': 'AA+',  'GBR': 'AA',   'DEU': 'AAA',  'FRA': 'AA-',  'JPN': 'A+',
    'CAN': 'AAA',  'AUS': 'AAA',  'CHE': 'AAA',  'NLD': 'AAA',  'SWE': 'AAA',
    'NOR': 'AAA',  'DNK': 'AAA',  'FIN': 'AA+',  'AUT': 'AA+',  'BEL': 'AA',
    'ITA': 'BBB',  'ESP': 'A',    'PRT': 'BBB+', 'GRC': 'BBB-', 'IRL': 'AA',
    'CHN': 'A+',   'KOR': 'AA',   'SGP': 'AAA',  'HKG': 'AA+',  'TWN': 'AA',
    'IND': 'BBB-', 'IDN': 'BBB',  'MYS': 'A-',   'THA': 'BBB+', 'PHL': 'BBB+',
    'VNM': 'BB+',  'BGD': 'BB-',  'PAK': 'CCC+', 'LKA': 'SD',   'NPL': 'NR',
    'BRA': 'BB-',  'MEX': 'BBB',  'COL': 'BB+',  'PER': 'BBB',  'CHL': 'A-',
    'ARG': 'CCC-', 'VEN': 'SD',   'ECU': 'B-',   'BOL': 'B+',   'PRY': 'BB',
    'URY': 'BBB',  'PAN': 'BBB',  'CRI': 'B+',   'DOM': 'BB',   'JAM': 'B+',
    'ZAF': 'BB-',  'EGY': 'B',    'NGA': 'B-',   'KEN': 'B',    'GHA': 'SD',
    'ETH': 'SD',   'AGO': 'B-',   'CIV': 'BB-',  'CMR': 'B',    'TZA': 'NR',
    'UGA': 'B+',   'SEN': 'B+',   'MAR': 'BB+',  'TUN': 'CCC',  'DZA': 'NR',
    'MOZ': 'CCC+', 'ZMB': 'SD',   'RWA': 'B+',   'MLI': 'NR',   'SDN': 'NR',
    'RUS': 'SD',   'TUR': 'B+',   'POL': 'A-',   'CZE': 'AA-',  'HUN': 'BBB',
    'ROU': 'BBB-', 'BGR': 'BBB',  'HRV': 'BBB+', 'SVN': 'AA-',  'SVK': 'A+',
    'UKR': 'SD',   'KAZ': 'BBB',  'AZE': 'BB+',  'UZB': 'BB-',  'GEO': 'BB',
    'SAU': 'A',    'ARE': 'AA',   'QAT': 'AA',   'KWT': 'AA',   'OMN': 'BB+',
    'BHR': 'B+',   'JOR': 'B+',   'ISR': 'AA-',  'LBN': 'SD',   'IRQ': 'B-',
    'NZL': 'AA',   'ISL': 'A',    'CYP': 'BBB',  'MLT': 'A-',   'EST': 'AA-',
    'LVA': 'A+',   'LTU': 'A',    'LUX': 'AAA',
}

# build a (iso3, year) -> rating dataframe using current rating as a static approximation
# in a production dataset you would maintain a full time-series per country
years_range = list(range(START_YEAR, END_YEAR + 1))
rating_rows = []
for iso3, rating in CURRENT_SP_RATINGS.items():
    for yr in years_range:
        rating_rows.append({
            'iso3_code': iso3, 'year': yr,
            'sp_credit_rating': rating,
            'sp_rating_numeric': RATING_MAP.get(rating, np.nan),
        })
ratings_df = pd.DataFrame(rating_rows)
print(f'Ratings reference: {ratings_df["iso3_code"].nunique()} countries')

In [ ]:
# ---- sovereign default events ----
# compiled from IMF, Standard & Poor's, and Reinhart & Rogoff "This Time Is Different" records
# format: (ISO3, year_of_default)
DEFAULT_EVENTS = [
    ('ARG', 2001), ('ARG', 2014), ('ARG', 2020),
    ('RUS', 2022),
    ('GRC', 2012),
    ('ECU', 2008), ('ECU', 2020),
    ('JAM', 2010),
    ('BLZ', 2006), ('BLZ', 2012), ('BLZ', 2017), ('BLZ', 2021),
    ('ZMB', 2020),
    ('SRB', 2004),
    ('MDG', 2002),
    ('CMR', 2004),
    ('ETH', 2023),
    ('GHA', 2022),
    ('LKA', 2022),
    ('VEN', 2017), ('VEN', 2018),
    ('URY', 2003),
    ('UKR', 2015),
    ('CIV', 2000), ('CIV', 2011),
    ('LBN', 2020),
    ('MOZ', 2017),
    ('SUR', 2020),
    ('PAK', 2023),  # IMF restructuring
]
defaults_df = pd.DataFrame(DEFAULT_EVENTS, columns=['iso3_code', 'year'])
defaults_df['sovereign_default_event'] = 1

# ---- IMF program (loan) flags ----
# each tuple is (ISO3, start_year, end_year) inclusive
# source: IMF MONA database https://mona.imf.org
IMF_PROGRAMS = [
    ('ARG', 2000, 2002), ('ARG', 2018, 2020), ('ARG', 2022, 2025),
    ('GRC', 2010, 2012), ('GRC', 2012, 2018),
    ('PRT', 2011, 2014),
    ('IRL', 2010, 2013),
    ('UKR', 2014, 2016), ('UKR', 2018, 2019), ('UKR', 2022, 2025),
    ('PAK', 2019, 2024),
    ('EGY', 2016, 2019), ('EGY', 2022, 2025),
    ('LKA', 2023, 2026),
    ('GHA', 2023, 2026),
    ('ETH', 2024, 2027),
    ('ZMB', 2022, 2025),
    ('TUN', 2013, 2015), ('TUN', 2023, 2024),
    ('JAM', 2010, 2012), ('JAM', 2013, 2016), ('JAM', 2019, 2021),
    ('KEN', 2021, 2023),
    ('SEN', 2021, 2023),
    ('CIV', 2023, 2025),
    ('ECU', 2019, 2022),
    ('CRI', 2021, 2023),
    ('BOL', 2003, 2004),
    ('IDN', 2000, 2003),
    ('RUS', 2000, 2002),
    ('TUR', 2002, 2008),
    ('BRA', 2003, 2005),
    ('ROU', 2009, 2011), ('ROU', 2011, 2013),
    ('HUN', 2008, 2010),
]
program_rows = []
for iso3, yr_start, yr_end in IMF_PROGRAMS:
    for yr in range(max(yr_start, START_YEAR), min(yr_end, END_YEAR) + 1):
        program_rows.append({'iso3_code': iso3, 'year': yr, 'imf_program_active': 1})
programs_df = pd.DataFrame(program_rows).drop_duplicates()

print(f'Default events:  {len(defaults_df)} records')
print(f'IMF programs:    {len(programs_df)} country-year records')

<h2 style="font-family: Georgia, serif; color: #1a1a2e; border-left: 4px solid #c0392b; padding-left: 12px;">5. Build the Main Panel</h2>
<p style="color: #555; font-size: 14px; max-width: 820px;">
  Merge all sources on (iso3_code, year), attach country metadata,
  and calculate the derived columns described in the dataset schema.
</p>

In [ ]:
# start from IMF panel and left-join everything in
panel = imf_panel.copy()

# add World Bank indicators
panel = panel.merge(wb_panel, on=['iso3_code', 'year'], how='outer')

# add country metadata from WB
panel = panel.merge(wb_meta, on='iso3_code', how='left')

# add S&P ratings
panel = panel.merge(ratings_df, on=['iso3_code', 'year'], how='left')

# add default events (fill missing with 0)
panel = panel.merge(defaults_df, on=['iso3_code', 'year'], how='left')
panel['sovereign_default_event'] = panel['sovereign_default_event'].fillna(0).astype(int)

# add IMF program flags
panel = panel.merge(programs_df, on=['iso3_code', 'year'], how='left')
panel['imf_program_active'] = panel['imf_program_active'].fillna(0).astype(int)

# add FRED US data if it was fetched
if not fred_panel.empty:
    panel = panel.merge(fred_panel, on=['iso3_code', 'year'], how='left')

# filter to real countries and sensible year range
panel = panel[
    panel['iso3_code'].str.len() == 3
    & panel['year'].between(START_YEAR, END_YEAR)
].copy()

panel = panel.sort_values(['country', 'year']).reset_index(drop=True)
print(f'Panel before derived cols: {len(panel):,} rows | {panel["iso3_code"].nunique()} countries')
panel.head(3)

In [ ]:
# ---- ensure every expected column exists before we start calculating ----
# if an API series failed silently, safe_merge_frames skipped it and the
# column is just absent - we add it as NaN so derived-column code never KeyErrors
EXPECTED_COLS = [
    'govt_debt_pct_gdp', 'govt_debt_usd_bn',
    'fiscal_balance_pct_gdp', 'primary_balance_pct_gdp',
    'govt_revenue_pct_gdp', 'interest_payments_pct_gdp',
    '_gdp_usd_bn', '_population_mn',
    'external_debt_usd_bn', 'foreign_reserves_usd_bn',
    '_gdp_wb_usd_bn', '_population_wb',
]
for col in EXPECTED_COLS:
    if col not in panel.columns:
        print(f'  adding missing column as NaN: {col}')
        panel[col] = np.nan


# ---- derived / calculated columns ----

# best available GDP estimate - prefer IMF, fall back to World Bank
panel['_gdp_bn'] = panel['_gdp_usd_bn'].combine_first(panel['_gdp_wb_usd_bn'])

# population in actual persons (IMF stores millions, WB stores actual persons)
panel['_pop'] = panel['_population_mn'].multiply(1e6).combine_first(panel['_population_wb'])

# debt per capita in USD
panel['debt_per_capita_usd'] = np.where(
    panel['_gdp_bn'].notna() & panel['govt_debt_pct_gdp'].notna() & panel['_pop'].notna(),
    (panel['_gdp_bn'] * 1e9 * panel['govt_debt_pct_gdp'] / 100) / panel['_pop'],
    np.nan,
)

# external debt as % of GDP
panel['external_debt_pct_gdp'] = np.where(
    panel['external_debt_usd_bn'].notna() & panel['_gdp_bn'].notna() & (panel['_gdp_bn'] > 0),
    panel['external_debt_usd_bn'] / panel['_gdp_bn'] * 100,
    np.nan,
)

# interest payments as % of government revenue (debt-trap warning signal)
panel['interest_pct_revenue'] = np.where(
    panel['interest_payments_pct_gdp'].notna() & panel['govt_revenue_pct_gdp'].notna()
    & (panel['govt_revenue_pct_gdp'] > 0),
    panel['interest_payments_pct_gdp'] / panel['govt_revenue_pct_gdp'] * 100,
    np.nan,
)

# reserves to external debt ratio (solvency buffer - higher is safer)
panel['reserves_to_external_debt'] = np.where(
    panel['foreign_reserves_usd_bn'].notna() & panel['external_debt_usd_bn'].notna()
    & (panel['external_debt_usd_bn'] > 0),
    panel['foreign_reserves_usd_bn'] / panel['external_debt_usd_bn'],
    np.nan,
)

# composite debt crisis risk score (0-100)
# 40% debt/GDP + 30% interest/revenue + 30% inverse reserves ratio
# missing values fall back to a neutral 50 so the score stays defined
debt_score = (panel['govt_debt_pct_gdp'].clip(0, 200) / 200 * 100).fillna(50)
int_score  = (panel['interest_pct_revenue'].clip(0, 50)  / 50  * 100).fillna(50)
res_raw    = panel['reserves_to_external_debt'].clip(0, 2).fillna(0.5)
res_score  = (1 - res_raw / 2) * 100  # low reserves -> high risk
panel['debt_crisis_risk_score'] = (
    0.40 * debt_score + 0.30 * int_score + 0.30 * res_score
).round(2)

# drop internal helper columns
drop_cols = [c for c in panel.columns if c.startswith('_')]
panel = panel.drop(columns=drop_cols)

# reorder columns to match the dataset schema
identity_cols = ['year', 'country', 'iso3_code', 'region', 'income_group']
debt_cols     = ['govt_debt_pct_gdp', 'govt_debt_usd_bn', 'external_debt_usd_bn',
                 'external_debt_pct_gdp', 'debt_per_capita_usd']
fiscal_cols   = ['fiscal_balance_pct_gdp', 'primary_balance_pct_gdp',
                 'govt_revenue_pct_gdp', 'interest_payments_pct_gdp',
                 'interest_pct_revenue', 'foreign_reserves_usd_bn',
                 'reserves_to_external_debt']
risk_cols     = ['sp_credit_rating', 'sp_rating_numeric', 'imf_program_active',
                 'imf_loan_usd_bn', 'sovereign_default_event', 'debt_crisis_risk_score']

# imf_loan_usd_bn is in the schema but has no automated source - placeholder NaN
panel['imf_loan_usd_bn'] = np.nan

fred_extra   = [c for c in ['us_10yr_yield_pct', 'us_federal_debt_pct_gdp_fred']
                if c in panel.columns]
ordered_cols = identity_cols + debt_cols + fiscal_cols + risk_cols + fred_extra
panel        = panel[[c for c in ordered_cols if c in panel.columns]]

print(f'Final panel: {len(panel):,} rows | {panel.shape[1]} columns')
panel.describe()

<h2 style="font-family: Georgia, serif; color: #1a1a2e; border-left: 4px solid #c0392b; padding-left: 12px;">6. ML Feature Engineering</h2>
<p style="color: #555; font-size: 14px; max-width: 820px;">
  Build a model-ready feature matrix with lag features, rolling-window statistics,
  year-over-year change rates, and a next-year prediction target.
  The target is whether <code>debt_crisis_risk_score</code> increases by more than 5 points
  in the following year (binary classification target).
</p>

In [ ]:
# columns we want to build lag/rolling features for
# filter to only what actually landed in the panel (some may be NaN-only if an API failed)
ALL_FEATURE_COLS = [
    'govt_debt_pct_gdp', 'fiscal_balance_pct_gdp', 'primary_balance_pct_gdp',
    'govt_revenue_pct_gdp', 'interest_payments_pct_gdp', 'interest_pct_revenue',
    'external_debt_pct_gdp', 'debt_per_capita_usd',
    'foreign_reserves_usd_bn', 'reserves_to_external_debt',
    'debt_crisis_risk_score', 'sp_rating_numeric',
]
FEATURE_COLS = [c for c in ALL_FEATURE_COLS if c in panel.columns]
print(f'Building features for {len(FEATURE_COLS)} columns: {FEATURE_COLS}')

ml      = panel.copy()
ml      = ml.sort_values(['iso3_code', 'year'])
grouped = ml.groupby('iso3_code')

for col in FEATURE_COLS:
    # 1-year and 2-year lags
    ml[f'{col}_lag1'] = grouped[col].shift(1)
    ml[f'{col}_lag2'] = grouped[col].shift(2)

    # 3-year and 5-year rolling means (computed on lagged values to avoid leakage)
    ml[f'{col}_roll3_mean'] = grouped[col].transform(lambda x: x.shift(1).rolling(3).mean())
    ml[f'{col}_roll5_mean'] = grouped[col].transform(lambda x: x.shift(1).rolling(5).mean())

    # 3-year rolling standard deviation (volatility signal)
    ml[f'{col}_roll3_std']  = grouped[col].transform(lambda x: x.shift(1).rolling(3).std())

    # year-over-year absolute change
    ml[f'{col}_yoy_change'] = grouped[col].diff(1)

    # year-over-year percentage change, capped at +/-200% to avoid explosion
    ml[f'{col}_yoy_pct']    = grouped[col].pct_change(1).clip(-2, 2) * 100


# ---- target variable ----
# 1 = debt_crisis_risk_score rises more than 5 points next year
if 'debt_crisis_risk_score' in panel.columns:
    next_score = grouped['debt_crisis_risk_score'].shift(-1)
    ml['target_risk_rise_next_yr'] = (
        (next_score - ml['debt_crisis_risk_score']) > 5
    ).astype('Int8')
else:
    ml['target_risk_rise_next_yr'] = pd.NA

ml = ml.sort_values(['iso3_code', 'year']).reset_index(drop=True)

print(f'ML feature matrix: {len(ml):,} rows | {ml.shape[1]} columns')
print(f'Feature columns added: {ml.shape[1] - panel.shape[1]}')

if ml['target_risk_rise_next_yr'].notna().any():
    print(f'\nTarget distribution (1 = risk rises next year):')
    print(ml['target_risk_rise_next_yr'].value_counts())

<h2 style="font-family: Georgia, serif; color: #1a1a2e; border-left: 4px solid #c0392b; padding-left: 12px;">7. Save Outputs</h2>
<p style="color: #555; font-size: 14px;">
  Write both CSVs to the working directory, then print a summary so the GitHub Actions log
  gives a clear confirmation of what was produced.
</p>

In [ ]:
# save main panel
panel.to_csv('sovereign_debt_panel.csv', index=False)
print(f'saved: sovereign_debt_panel.csv')

# save ML features
ml.to_csv('ml_features.csv', index=False)
print(f'saved: ml_features.csv')

# quick stats summary
print('\n--- sovereign_debt_panel.csv ---')
print(f'  rows:       {len(panel):,}')
print(f'  columns:    {panel.shape[1]}')
print(f'  countries:  {panel["iso3_code"].nunique()}')
print(f'  year range: {panel["year"].min()} - {panel["year"].max()}')
print(f'  debt/GDP coverage: {panel["govt_debt_pct_gdp"].notna().sum()} rows')

print('\n--- ml_features.csv ---')
print(f'  rows:    {len(ml):,}')
print(f'  columns: {ml.shape[1]}')
print(f'  target (not null): {ml["target_risk_rise_next_yr"].notna().sum()}')

In [ ]:
# ---- generate README.md ----
# the workflow expects this file to exist alongside the CSVs

now_str   = datetime.utcnow().strftime('%B %d, %Y')
row_count = len(panel)
countries = panel['iso3_code'].nunique()
yr_min    = int(panel['year'].min())
yr_max    = int(panel['year'].max())

readme = textwrap.dedent(f"""\
# Global Sovereign Debt Crisis Monitor ({yr_min}-Present)

Last updated: {now_str} (auto-generated by notebook.ipynb via GitHub Actions)

A yearly country-level panel covering sovereign debt levels, fiscal health indicators,
and early-warning crisis signals for {countries} countries from {yr_min} to {yr_max}.
Built to support both exploratory analysis and machine learning tasks.

---

## Files

| File | Rows | Description |
| ---- | ---- | ----------- |
| sovereign_debt_panel.csv | {row_count:,} | Main analysis file - all indicators merged |
| ml_features.csv | {len(ml):,} | Lags, rolling windows, YoY changes, and prediction target |

---

## Data Sources

- **IMF World Economic Outlook** - DataMapper API, no key needed
  - Series: GGXWDG_NGDP, GGXWDG, GGXCNL_NGDP, GGXONLB_NGDP, GGR_NGDP, GGXI_NGDP
- **World Bank Open Data** - REST API, no key needed
  - Indicators: DT.DOD.DECT.CD, FI.RES.TOTL.CD, NY.GDP.MKTP.CD, SP.POP.TOTL
- **FRED** (St. Louis Fed) - US Treasury yields, US federal debt. Free API key at fred.stlouisfed.org
- **Manual reference** - S&P ratings, sovereign default events, IMF program flags

---

## Column Reference - sovereign_debt_panel.csv

### Identity

| Column | Type | Description |
| ------ | ---- | ----------- |
| year | int | Calendar year |
| country | text | Country name (standardised via World Bank) |
| iso3_code | text | ISO 3166-1 alpha-3 code |
| region | text | World Bank region (South Asia, MENA, ...) |
| income_group | text | High / Upper-middle / Lower-middle / Low |

### Debt Burden

| Column | Type | Description | Source |
| ------ | ---- | ----------- | ------ |
| govt_debt_pct_gdp | float | Gross govt debt as % of GDP | IMF: GGXWDG_NGDP |
| govt_debt_usd_bn | float | Gross govt debt in USD billions | IMF: GGXWDG |
| external_debt_usd_bn | float | External debt owed to foreign creditors | WB: DT.DOD.DECT.CD |
| external_debt_pct_gdp | float | External debt as % of GDP | Calculated |
| debt_per_capita_usd | float | Govt debt per person in USD | Calculated |

### Fiscal Health

| Column | Type | Description | Source |
| ------ | ---- | ----------- | ------ |
| fiscal_balance_pct_gdp | float | Revenue minus spending as % of GDP (negative = deficit) | IMF: GGXCNL_NGDP |
| primary_balance_pct_gdp | float | Fiscal balance before interest payments | IMF: GGXONLB_NGDP |
| govt_revenue_pct_gdp | float | Total government revenue as % of GDP | IMF: GGR_NGDP |
| interest_payments_pct_gdp | float | Annual interest paid on debt as % of GDP | IMF: GGXI_NGDP |
| interest_pct_revenue | float | Interest payments / revenue - a debt-trap warning signal | Calculated |
| foreign_reserves_usd_bn | float | Foreign exchange reserves in USD billions | WB: FI.RES.TOTL.CD |
| reserves_to_external_debt | float | Reserves divided by external debt - solvency buffer | Calculated |

### Risk Signals

| Column | Type | Description | Source |
| ------ | ---- | ----------- | ------ |
| sp_credit_rating | text | S&P sovereign rating (AAA to D) | Manual / S&P |
| sp_rating_numeric | int | Rating mapped to numbers: AAA=21, D=0 | Mapped |
| imf_program_active | bool (0/1) | Country is under an active IMF loan program | IMF MONA |
| imf_loan_usd_bn | float | IMF disbursement amount (USD billions) | IMF MONA |
| sovereign_default_event | bool (0/1) | A default event occurred this year | Public records |
| debt_crisis_risk_score | float | Composite 0-100 risk score | Calculated |

---

## Risk Score Methodology

The `debt_crisis_risk_score` is a 0-100 composite built from three components:

- **40%** - Debt/GDP (capped at 200%, normalized to 0-100)
- **30%** - Interest payments as % of revenue (capped at 50%)
- **30%** - Inverse of the reserves-to-external-debt ratio (low reserves = high risk)

A score above 65 generally corresponds to countries that have experienced recent debt distress.
This is a rough heuristic - use it as one signal, not as a definitive judgment.

---

## ML Features (ml_features.csv)

All numeric indicators above get five additional derived columns per variable:

- `*_lag1`, `*_lag2` - previous 1 and 2 year values
- `*_roll3_mean`, `*_roll5_mean` - 3 and 5 year rolling averages
- `*_roll3_std` - 3 year rolling standard deviation (volatility)
- `*_yoy_change` - absolute year-over-year change
- `*_yoy_pct` - percentage year-over-year change (capped at +/-200%)

The **target column** is `target_risk_rise_next_yr` (1 = risk score increases more than 5 points
the following year, 0 = it does not). Rows where the next year is not yet available have NaN here.

---

## License

CC0 1.0 Universal - public domain. Use it however you want.

IMF and World Bank data are published under their respective open data policies.
""")

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme)

print('saved: README.md')
print(f'README length: {len(readme)} characters')